# N5 - Faulted-zone identification (the task between "type" and "location")

**Question.** The paper classifies fault *type*. Every compared work (C10, E03, E09, F08)
also outputs a location of some granularity. A reviewer will ask why we localize nothing.
This notebook tests whether **coarse zone identification** — which feeder section /
lateral contains the fault — is achievable from the same voltage snapshot, at the same sensor
budget, and whether that ability is what the placement is actually buying.

**Why a zone, not a distance.** Exact distance needs impedance-based ranging, which our
single-snapshot voltage-only input does not support. A *zone* is a discrete label over a
graph partition, so it is a fair extension of the same classifier machinery and it matches
what C10 (faulted bus), E09 (32 lines) and E03 (line section) report.

**Zone construction.** The feeder is partitioned into zones by graph distance from the
substation (BFS depth). `n_zones` is varied so accuracy-vs-granularity is a curve, not a
single number — this is what makes it informative rather than a vanity result.

In [ ]:
# ============================================================================
#  SETUP - run this cell first. Change SMOKE and, if you like, N_JOBS.
# ============================================================================
SMOKE = False         # True: ~1 feeder, 1 seed, small N, minutes. False: full run.

# Optional: restrict this execution to a subset of work units. Leave empty for everything.
# Set e.g. ONLY_FEEDERS = [123] to run just the largest feeder, or ONLY_KS = [2, 3] to run a
# slice of the k grid. Every notebook is resumable, so a sliced run can be continued later
# without redoing finished units (state lives in /kaggle/working/results).
ONLY_FEEDERS = []
ONLY_KS = []

import os, sys, json, time, platform, warnings
warnings.filterwarnings("ignore")
t_start = time.time()

# Where the attached Kaggle dataset is mounted. Everything we need (code, .npz data and
# the precomputed design grids) lives under this one directory.
import glob as _g
_c = sorted(_g.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
DATA = os.path.dirname(_c[0]) if _c else "/kaggle/input/snapshot-fd-code-data/experiments"
print("DATA =", DATA)
if not os.path.isdir(DATA):
    # local fallback so the notebook also runs on a laptop for a smoke test
    DATA = os.environ.get("K4_DATA", DATA)
assert os.path.isdir(DATA), (
    "Attach the dataset 'snapshot-fd-code-data' (Add input -> Your datasets). "
    "Expected /kaggle/input/snapshot-fd-code-data/experiments")
# The dataset root also carries k4lib.py and e01.py next to experiments/.
# On Kaggle that is /kaggle/input/snapshot-fd-code-data; locally the task folder.
ROOT = os.environ.get("K4_ROOT", os.path.dirname(DATA))
for p in (DATA, ROOT):
    if p and p not in sys.path:
        sys.path.insert(0, p)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results")
os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS          # k4lib reads this
os.environ["K4_DATA"] = DATA

import numpy as np

# OpenDSS is not needed (design grids are precomputed); placement.py imports simlib,
# which imports opendssdirect at load time. Provide an empty stand-in if it is missing.
try:
    import opendssdirect  # noqa: F401
except ImportError:
    import types as _t
    sys.modules["opendssdirect"] = _t.ModuleType("opendssdirect")

import k4lib as K
import features as F
import placement as P
import placement2 as Q

# forward the optional work-slicing from the SETUP cell into the library
K.ONLY_FEEDERS = list(ONLY_FEEDERS)
K.ONLY_KS = list(ONLY_KS)

print("=== environment ===")
K.print_env()
print("SMOKE =", SMOKE)
if ONLY_FEEDERS:
    print("ONLY_FEEDERS =", ONLY_FEEDERS)
if ONLY_KS:
    print("ONLY_KS =", ONLY_KS)
print("elapsed", round(time.time() - t_start, 1), "s")

## 1. Zone labels from graph topology

A zone is a set of buses at equal (or near-equal) BFS depth from the substation. This is a
defensible physical proxy: protection zones in distribution networks are also defined by
topological sections, not by Euclidean distance.

In [ ]:
from collections import deque

def bfs_depth(edges, n_bus, source):
    adj = [[] for _ in range(n_bus)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    d = np.full(n_bus, -1)
    d[source] = 0
    q = deque([source])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if d[v] < 0:
                d[v] = d[u] + 1
                q.append(v)
    return d

def quantile_zones(depth, n_zones):
    """Bin buses into n_zones groups of roughly equal size by BFS depth."""
    d = depth.astype(float)
    edges_ = np.quantile(d, np.linspace(0, 1, n_zones + 1)[1:-1])
    return np.digitize(d, edges_)

feeders = K.feeder_list(SMOKE)
NZONES = [2, 3] if SMOKE else [2, 3, 4, 5, 6, 8]
zone_info = {}
for nb in feeders:
    ds = K.dataset(nb)
    edges, order = K.feeder_edges(nb)
    src = ds.buses.index(ds.substation) if ds.substation in ds.buses else 0
    depth = bfs_depth(edges, len(ds.buses), src)
    zone_info[nb] = (ds, edges, depth)
    print(f"IEEE-{nb}: {len(ds.buses)} buses, depth {depth.min()}..{depth.max()}, "
          f"reachable {int((depth >= 0).sum())}")
    for nz in NZONES:
        z = quantile_zones(np.maximum(depth, 0), nz)
        counts = np.bincount(z, minlength=nz)
        print(f"   {nz} zones -> sizes {counts.tolist()}")

## 2. Zone classification vs number of zones and sensor count

The comparison that matters: how does zone accuracy degrade as the partition gets finer, and
does adding sensors help more for zones than for types? If zones need many more sensors, that
is the honest reason our paper does not claim localization.

In [ ]:
rows = []
for nb in feeders:
    ds, edges, depth = zone_info[nb]
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
    # fault bus -> zone; samples are labelled by their fault bus
    bus_pos = {b: i for i, b in enumerate(ds.buses)}
    fb_idx = np.array([bus_pos[b] for b in ds.fbus])
    for nz in NZONES:
        z = quantile_zones(np.maximum(depth, 0), nz)
        yz = z[fb_idx]
        # drop zones with fewer than 25 samples (can't cross-validate)
        cnt = np.bincount(yz, minlength=nz)
        keep = cnt >= 25
        m_ok = keep[yz]
        if m_ok.sum() < 100:
            print(f"IEEE-{nb} nz={nz}: too few samples, skipped")
            continue
        for k in K.k_list(nb, SMOKE):
            obs, _ = Q.dmp2(m2, wgt, k)
            for noise in ("1% TVE", "uPMU"):
                X = K.features_of(ds, obs, 0, noise, "FPS")[m_ok]
                y = yz[m_ok]
                # reindex labels to be contiguous
                u = np.unique(y)
                yr = np.searchsorted(u, y)
                r = K.folds_acc(X, yr, seeds=(0,) if SMOKE else (0, 1))
                rows.append(dict(feeder=nb, n_zones=nz, k=k, noise=noise,
                                 n=int(m_ok.sum()), chance=1.0 / len(u),
                                 acc=r["acc"], acc_sd=r["acc_sd"]))
                print(f"IEEE-{nb} {nz} zones k={k} {noise:8s} "
                      f"acc {r['acc']*100:5.1f}%  chance {100/len(u):4.1f}%  n={int(m_ok.sum())}")

## 3. Type and zone jointly: does one placement serve both?

A single sensor set is only operationally useful if it does both jobs. This cell trains on
the fault type and on the zone from the *same* features and reports both, which is the fair
statement of what the paper's placement delivers.

In [ ]:
joint = []
for nb in feeders:
    ds, edges, depth = zone_info[nb]
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
    bus_pos = {b: i for i, b in enumerate(ds.buses)}
    fb_idx = np.array([bus_pos[b] for b in ds.fbus])
    nz = 3
    yz = quantile_zones(np.maximum(depth, 0), nz)[fb_idx]
    u = np.unique(yz)
    yr = np.searchsorted(u, yz)
    for k in K.k_list(nb, SMOKE):
        obs, _ = Q.dmp2(m2, wgt, min(k, len(buses) - 1))
        for noise in ("1% TVE", "uPMU"):
            X = K.features_of(ds, obs, 0, noise, "FPS")
            rt = K.folds_acc(X, ds.y, seeds=(0,) if SMOKE else (0, 1))
            rz = K.folds_acc(X, yr, seeds=(0,) if SMOKE else (0, 1))
            joint.append(dict(feeder=nb, k=k, noise=noise, n_zones=nz,
                              acc_type=rt["acc"], acc_zone=rz["acc"],
                              chance_zone=1.0 / len(u)))
            print(f"IEEE-{nb} k={k} {noise:8s} type {rt['acc']*100:5.1f}%  "
                  f"zone {rz['acc']*100:5.1f}% (chance {100/len(u):4.1f}%)")

## 4. Save

In [ ]:
import pandas as pd
pd.DataFrame(rows).to_csv(os.path.join(WORK, "N5_zones.csv"), index=False)
pd.DataFrame(joint).to_csv(os.path.join(WORK, "N5_joint.csv"), index=False)
for nb in feeders:
    K.save_unit("n5", {"feeder": nb},
                dict(rows=[r for r in rows if r["feeder"] == nb],
                     joint=[r for r in joint if r["feeder"] == nb], meta=K.meta()))
print("TOTAL", round((time.time()-t_start)/60, 1), "min")